## Read CSV files in volume, stream all artist csvs tagged with v*


In [0]:
import pyspark.sql.functions as F
files = dbutils.fs.ls("/Volumes/spotify-data-project-dev/default/spotify-data-project")
display(files)

path,name,size,modificationTime
dbfs:/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_albums.csv,spotify_albums.csv,2631253,1787223670000
dbfs:/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_artist_features.csv,spotify_artist_features.csv,12015713,1787223675000
dbfs:/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_artists.csv,spotify_artists.csv,8264572,1787223673000
dbfs:/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_audio_features.csv,spotify_audio_features.csv,270402495,1787223708000
dbfs:/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_decade_trends.csv,spotify_decade_trends.csv,1137,1787223668000
dbfs:/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_features.csv,spotify_features.csv,456178368,1787223708000
dbfs:/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_genre_tags.csv,spotify_genre_tags.csv,142851623,1787223708000
dbfs:/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_playlist_cooccurrence.csv,spotify_playlist_cooccurrence.csv,191578962,1787223708000
dbfs:/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_popularity_metrics.csv,spotify_popularity_metrics.csv,175451915,1787223708000
dbfs:/Volumes/spotify-data-project-dev/default/spotify-data-project/taste_cluster_report.csv,taste_cluster_report.csv,23446,1787223668000


In [0]:
df_raw = spark.read.csv("/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_artists.csv", header=True)
display(df_raw.limit(10))
display(df_raw.columns)

artist_id,name,name_normalized,country_of_origin,career_start_year,primary_genre_l1,primary_genre_l2,spotify_followers,spotify_popularity,monthly_listeners,musicbrainz_id,lastfm_listeners,lastfm_play_count,is_solo_artist,source_dataset
artist:000347dd-e0b0-a87a-8638-ea93e2fa1653,Baskerville,baskerville,null,2017,Metal,Heavy Metal,null,null,null,null,null,null,null,compiled_multi_source
artist:00048daf-510a-51ec-64c9-a347cc29b997,Shion Tsuji,shion tsuji,null,2009,Folk,Singer-Songwriter,null,null,null,null,null,null,null,compiled_multi_source
artist:00057abd-bcac-477a-074a-7e93e217674e,Yu Takahashi,yu takahashi,null,2010,Folk,Acoustic,null,null,null,null,null,null,null,compiled_multi_source
artist:0006978c-83ee-4981-1dd4-764bc9b27ef4,Jimmy Gourley,jimmy gourley,null,2004,Instrumental,Guitar,null,null,null,null,null,null,null,compiled_multi_source
artist:00080783-707a-2414-a10e-00aa1c54622a,J Bas Y Santy,j bas y santy,null,2019,Latin,Latin,null,null,null,null,null,null,null,compiled_multi_source
artist:0008fec2-3531-0f7c-12b9-1a7563bdc77e,TK N Cash,tk n cash,null,2014,Hip-Hop,Hip-Hop,null,null,null,null,null,null,null,compiled_multi_source
artist:00090478-00b5-21c8-65c4-c4f308a979c9,Tango Siempre,tango siempre,null,2008,Latin,Tango,null,null,null,null,null,null,null,compiled_multi_source
artist:0009cfc6-a2f4-d888-2beb-aa5530427829,Kideko,kideko,null,2015,Electronic,Deep House,null,null,null,null,null,null,null,compiled_multi_source
artist:000a2f01-80fd-07f9-29d1-9c86e08a6bd1,Chris Young,chris young,null,2006,Country,Country,2918550.0,65.0,null,null,null,null,null,compiled_multi_source
artist:000c69c5-05fa-6523-313e-e2387ee41f4b,Lonely God,lonely god,null,2018,Pop,Indie Pop,null,null,null,null,null,null,null,compiled_multi_source


_1
artist_id
name
name_normalized
country_of_origin
career_start_year
primary_genre_l1
primary_genre_l2
spotify_followers
spotify_popularity
monthly_listeners


In [0]:
try:
    dbutils.fs.rm("/Volumes/spotify-data-project-dev/default/checkpoints/read_v3", True)
    dbutils.fs.rm("/Volumes/spotify-data-project-dev/default/checkpoints/write_artists_v3", True)
    print("Cleared checkpoint directories")
except Exception as e:
    print(f"Could not clear checkpoints: {e}")

Cleared checkpoint directories


## Define streaming bronze table, list of artists that could be updated every month 

In [0]:
SOURCE_PATH = (
    "/Volumes/spotify-data-project-dev/default/"
    "spotify-data-project/"
)

SCHEMA_PATH = (
    "/Volumes/spotify-data-project-dev/default/"
    "checkpoints/artists_schema"
)

CHECKPOINT_PATH = (
    "/Volumes/spotify-data-project-dev/default/"
    "checkpoints/write_artists_v3"
)


def process_bronze_artist():
    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", SCHEMA_PATH)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("pathGlobFilter", "spotify_artists_*.csv")
        .option("header", "true")
        .load(SOURCE_PATH)
        .drop("_rescued_data")
        .withColumn("source_file", F.col("_metadata.file_name"))
        .withColumn("timestamp_added", F.current_timestamp())
        .withColumn(
            "year_month",
            F.date_format("timestamp_added", "yyyy-MM")
        )
    )

    query = (
        df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .trigger(availableNow=True)
        .toTable("`spotify-data-project-dev`.`default`.bronze_artists")
    )

    query.awaitTermination()


process_bronze_artist()

In [0]:
%sql 
select * from bronze_artists limit 10 

artist_id,name,name_normalized,country_of_origin,career_start_year,primary_genre_l1,primary_genre_l2,spotify_followers,spotify_popularity,monthly_listeners,musicbrainz_id,lastfm_listeners,lastfm_play_count,is_solo_artist,source_dataset,source_file,timestamp_added,year_month
artist:000347dd-e0b0-a87a-8638-ea93e2fa1653,Baskerville,baskerville,null,2017,Metal,Heavy Metal,null,null,null,null,null,null,null,compiled_multi_source,spotify_artists_v1.csv,2026-08-21T06:58:47.126Z,2026-08
artist:00048daf-510a-51ec-64c9-a347cc29b997,Shion Tsuji,shion tsuji,null,2009,Folk,Singer-Songwriter,null,null,null,null,null,null,null,compiled_multi_source,spotify_artists_v1.csv,2026-08-21T06:58:47.126Z,2026-08
artist:00057abd-bcac-477a-074a-7e93e217674e,Yu Takahashi,yu takahashi,null,2010,Folk,Acoustic,null,null,null,null,null,null,null,compiled_multi_source,spotify_artists_v1.csv,2026-08-21T06:58:47.126Z,2026-08
artist:0006978c-83ee-4981-1dd4-764bc9b27ef4,Jimmy Gourley,jimmy gourley,null,2004,Instrumental,Guitar,null,null,null,null,null,null,null,compiled_multi_source,spotify_artists_v1.csv,2026-08-21T06:58:47.126Z,2026-08
artist:00080783-707a-2414-a10e-00aa1c54622a,J Bas Y Santy,j bas y santy,null,2019,Latin,Latin,null,null,null,null,null,null,null,compiled_multi_source,spotify_artists_v1.csv,2026-08-21T06:58:47.126Z,2026-08
artist:0008fec2-3531-0f7c-12b9-1a7563bdc77e,TK N Cash,tk n cash,null,2014,Hip-Hop,Hip-Hop,null,null,null,null,null,null,null,compiled_multi_source,spotify_artists_v1.csv,2026-08-21T06:58:47.126Z,2026-08
artist:00090478-00b5-21c8-65c4-c4f308a979c9,Tango Siempre,tango siempre,null,2008,Latin,Tango,null,null,null,null,null,null,null,compiled_multi_source,spotify_artists_v1.csv,2026-08-21T06:58:47.126Z,2026-08
artist:0009cfc6-a2f4-d888-2beb-aa5530427829,Kideko,kideko,null,2015,Electronic,Deep House,null,null,null,null,null,null,null,compiled_multi_source,spotify_artists_v1.csv,2026-08-21T06:58:47.126Z,2026-08
artist:000a2f01-80fd-07f9-29d1-9c86e08a6bd1,Chris Young,chris young,null,2006,Country,Country,2918550.0,65.0,null,null,null,null,null,compiled_multi_source,spotify_artists_v1.csv,2026-08-21T06:58:47.126Z,2026-08
artist:000c69c5-05fa-6523-313e-e2387ee41f4b,Lonely God,lonely god,null,2018,Pop,Indie Pop,null,null,null,null,null,null,null,compiled_multi_source,spotify_artists_v1.csv,2026-08-21T06:58:47.126Z,2026-08


## Set-up ALBUM static table 

In [0]:
df_raw = spark.read.csv("/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_albums.csv", header=True)
display(df_raw.limit(10))
display(df_raw.columns)

album_id,title,artist_id,album_type,release_year,release_date,track_count,label,upc,spotify_id,musicbrainz_id,source_dataset
000YOrgQoB5IiiH95Yb8vY,The Road,artist:5e640c3e-4a97-4028-cc7f-3b2209ebcb6d,album,2019,null,1,null,null,000YOrgQoB5IiiH95Yb8vY,null,tidy_tuesday_dbcl
000f3dTtvpazVzv35NuZmn,"Make It Fast, Make It Slow (Soundway Records)",artist:760061f6-bfde-75c2-9af1-2f252d4d3345,album,2012,null,1,null,null,000f3dTtvpazVzv35NuZmn,null,tidy_tuesday_dbcl
006AgHXrEw13oyg0D8evRa,Bad Boy Greatest Hits Vol. 1,artist:7f6ffaa6-bb0b-4080-17b6-2254211691b5,album,1998,null,1,null,null,006AgHXrEw13oyg0D8evRa,null,tidy_tuesday_dbcl
006XcK33PV5wCV9g629nYA,A.E.I.O.U.R.E.M.I.X.E.S,artist:321c84ad-55f9-0817-8779-9b087216a2ae,album,2013,null,1,null,null,006XcK33PV5wCV9g629nYA,null,tidy_tuesday_dbcl
006qPzTTwYGb4t4glyTaLw,Bratva iv - Hololo,artist:60089ca0-c506-a23b-4138-2ab9ec655173,album,2019,null,1,null,null,006qPzTTwYGb4t4glyTaLw,null,tidy_tuesday_dbcl
006wfSK6TYpZIFU9yEzPEo,Let Me Down,artist:41e65597-a7bd-9945-97fb-b98cf6779c19,album,2018,null,1,null,null,006wfSK6TYpZIFU9yEzPEo,null,tidy_tuesday_dbcl
007354CLF8nvIbRN7Gwns9,Different Place,artist:de8951d7-767a-98d1-5ef0-65faa35d284b,album,2018,null,1,null,null,007354CLF8nvIbRN7Gwns9,null,tidy_tuesday_dbcl
008YlPoRP6UXxI7L5rKzuZ,Into The Wild,artist:bf73d632-dc6d-a534-3551-1440f4c43e87,album,2015,null,1,null,null,008YlPoRP6UXxI7L5rKzuZ,null,tidy_tuesday_dbcl
00AVS4xDGxD61LgiIkDaN7,OTRA COSA,artist:43f2caee-e6d1-ecec-2ec7-8d7d9adee41d,album,2019,null,2,null,null,00AVS4xDGxD61LgiIkDaN7,null,tidy_tuesday_dbcl
00DKwMpzeC9SVGOJ7NQrnp,Aquaman,artist:a9418fa3-7bfb-b7c5-a79c-b29e524cc6a6,album,2020,null,1,null,null,00DKwMpzeC9SVGOJ7NQrnp,null,tidy_tuesday_dbcl


_1
album_id
title
artist_id
album_type
release_year
release_date
track_count
label
upc
spotify_id


In [0]:
def process_bronze_albums(): 
    df_albums = spark.read.csv(
        "/Volumes/spotify-data-project-dev/default/spotify-data-project/spotify_albums.csv",
        header=True,
        inferSchema=True
    ) \
        .withColumn("timestamp_added", F.current_timestamp()) \
        .withColumn("year_month", F.date_format("timestamp_added", "yyyy-MM"))
    
    df_albums.write \
        .mode("append") \
        .partitionBy("release_year") \
        .saveAsTable("`spotify-data-project-dev`.`default`.bronze_albums")
    
    print(f"Wrote {df_albums.count()} rows to bronze_albums")

process_bronze_albums()

Wrote 18355 rows to bronze_albums
